In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def show(images):
    plt.figure(figsize=(4 * len(images), 4))
    for i, (title, im) in enumerate(images.items()):
        plt.subplot(1, len(images), i + 1)
        if im.ndim == 2:
            plt.imshow(im, cmap="gray")
        else:
            plt.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.xticks([]), plt.yticks([])
    plt.tight_layout()
    plt.show()

In [ ]:
img_path = "../../images/lenna.png"
img = cv.imread(img_path, 0)

show({"Original Image": img})

1. Spatial Domain Filtering
a. Low-pass filtering using box filter
b. Low-pass filtering using gaussian filter

A low-pass filter smooths an image by replacing each pixel with a weighted average of its
neighbours, which removes high-frequency detail such as noise and sharp edges.
- Box filter: every neighbour gets an equal weight $1/n^2$.
- Gaussian filter: neighbours are weighted by a Gaussian bell centred on the pixel, so closer
  pixels count more.

Both are just "convolve the image with a kernel", done here with `cv.filter2D` - a box kernel for
the box filter, and a Gaussian kernel from `cv.getGaussianKernel` for the Gaussian filter.

In [ ]:
box_kernel = np.ones((5, 5)) / 25  # larger kernel size -> stronger smoothing, more blur
gauss_kernel = cv.getGaussianKernel(5, 1) @ cv.getGaussianKernel(5, 1).T  # larger sigma -> wider spread, more blur

box_lpf = cv.filter2D(img, -1, box_kernel)
gaussian_lpf = cv.filter2D(img, -1, gauss_kernel)

show({"Original": img, "Box LPF": box_lpf, "Gaussian LPF": gaussian_lpf})

c. High-pass filtering
d. Unsharp masking
e. High-boost filter

Any image is the sum of a low-frequency part and a high-frequency part, so the high-frequency
(edge/detail) content can be recovered as $\text{high-pass} = f - \bar f$, where $\bar f$ is a
blurred (low-pass) version of the image $f$.
- Unsharp masking adds this detail back once to sharpen edges: $g = f + (f - \bar f)$.
- High-boost filtering generalizes this with a gain $k > 1$: $g = f + k(f - \bar f)$, emphasizing
  edges more strongly than plain unsharp masking.

In [ ]:
blurred = cv.filter2D(img, -1, gauss_kernel)
mask = cv.subtract(img, blurred)

# High-pass = original - low-pass
highpass = mask

# Unsharp masking: sharpened = original + (original - blurred)
unsharp = cv.add(img, mask)

# High-boost: sharpened = original + k * (original - blurred), k > 1
k = 2.5  # k > 1 boosts edges more strongly; k=1 is plain unsharp masking, too large a k over-sharpens and introduces halos
highboost = np.clip(img + k * mask, 0, 255).astype(np.uint8)

show({
    "Original": img,
    "Blurred": blurred,
    "High-pass": highpass,
    "Unsharp Masking": unsharp,
    "High-boost (k=2.5)": highboost,
})

2. Transform domain-based image enhancement
a. Generation of low-pass filter mask
   i. Ideal low-pass filter
   ii. Butterworth filter
   iii. Gaussian filter

Filtering in the frequency domain means multiplying the image's Fourier transform $F(u, v)$ by a
mask $H(u, v)$ that keeps frequencies close to the centre (distance $D(u, v)$ from the origin is
small) and attenuates the rest:
- Ideal: $H = 1$ if $D \le D_0$, else $0$ - a sharp cutoff that causes ringing.
- Butterworth: $H = \dfrac{1}{1 + (D / D_0)^{2n}}$ - a smoother, adjustable roll-off.
- Gaussian: $H = e^{-D^2 / 2D_0^2}$ - the smoothest roll-off, with no ringing.

In [ ]:
M, N = img.shape

# Distance of every point from the centre
u, v = np.meshgrid(np.arange(N) - N // 2, np.arange(M) - M // 2)
D = np.sqrt(u**2 + v**2)

d0, n = 30, 2  # larger d0 keeps more high frequencies -> sharper, less blurred result; larger n makes Butterworth's roll-off sharper (closer to ideal)

ideal_mask = (D <= d0).astype(np.float32)
butter_mask = 1 / (1 + (D / d0) ** (2 * n))
gauss_mask = np.exp(-(D**2) / (2 * d0**2))

F = np.fft.fftshift(np.fft.fft2(img))


def apply_mask(mask):
    filtered = np.fft.ifft2(np.fft.ifftshift(F * mask))
    return np.clip(np.abs(filtered), 0, 255).astype(np.uint8)


show({
    "Ideal LPF Mask": (ideal_mask * 255).astype(np.uint8),
    "Butterworth LPF Mask": (butter_mask * 255).astype(np.uint8),
    "Gaussian LPF Mask": (gauss_mask * 255).astype(np.uint8),
    "Original": img,
    "Ideal LPF": apply_mask(ideal_mask),
    "Butterworth LPF": apply_mask(butter_mask),
    "Gaussian LPF": apply_mask(gauss_mask),
})

b. Generation of high-pass filter mask
   i. Ideal high-pass filter
   ii. Butterworth filter
   iii. Gaussian filter

Low-frequency content and high-frequency content together make up the whole spectrum, so a
high-pass mask is simply what its matching low-pass mask removes: $H_{hp} = 1 - H_{lp}$.

In [ ]:
# A high-pass mask is simply 1 - (the matching low-pass mask)
ideal_h = 1 - ideal_mask
butter_h = 1 - butter_mask
gauss_h = 1 - gauss_mask

show({
    "Ideal HPF Mask": (ideal_h * 255).astype(np.uint8),
    "Butterworth HPF Mask": (butter_h * 255).astype(np.uint8),
    "Gaussian HPF Mask": (gauss_h * 255).astype(np.uint8),
    "Original": img,
    "Ideal HPF": apply_mask(ideal_h),
    "Butterworth HPF": apply_mask(butter_h),
    "Gaussian HPF": apply_mask(gauss_h),
})

c. High-frequency emphasis filter in frequency domain

High-frequency emphasis boosts edges/detail while still retaining some of the low-frequency
(average brightness) information, using $H = a + b \cdot H_{hp}$ with an offset $a \ge 0$ and a
boost factor $b > 1$.

In [ ]:
a_const, b_const = 0.5, 2.0  # larger a keeps more of the low-frequency base image; larger b boosts edge/detail more strongly

emphasis_mask = a_const + b_const * gauss_h

show({"Original": img, "High-Frequency Emphasis": apply_mask(emphasis_mask)})

d. Ideal band-pass filter in the frequency domain
e. Ideal band-reject filter in the frequency domain

A band-reject filter zeroes out a ring of frequencies between $D_0 - W/2$ and $D_0 + W/2$
(useful for removing periodic noise at a known frequency), while a band-pass filter keeps only
that ring: $H_{bp} = 1 - H_{br}$.

In [ ]:
d0_band, w = 40, 20  # d0_band sets which frequency band is removed/kept; larger w widens the band of affected frequencies

bandreject_mask = np.ones((M, N), dtype=np.float32)
bandreject_mask[(D >= d0_band - w / 2) & (D <= d0_band + w / 2)] = 0
bandpass_mask = 1 - bandreject_mask

show({
    "Original": img,
    "Band Pass Mask": bandpass_mask,
    "Ideal Band-Pass": apply_mask(bandpass_mask),
    "Ideal Band-Reject": apply_mask(bandreject_mask),
})

3. Homomorphic filter

An image can be modeled as illumination times reflectance, $f = i \cdot r$, where illumination
varies slowly (low frequency) and reflectance carries the fine detail (high frequency). Taking
the log turns this product into a sum, $\ln f = \ln i + \ln r$, so an ordinary frequency filter
can now suppress illumination and boost reflectance:
$H = (\gamma_H - \gamma_L)\left(1 - e^{-D^2/D_0^2}\right) + \gamma_L$, with $\gamma_L < 1$ and
$\gamma_H > 1$. Exponentiating after the inverse FFT undoes the initial log.

In [ ]:
gamma_l, gamma_h, c, d0 = 0.5, 2.0, 1, 30  # gamma_l<1 suppresses illumination; gamma_h>1 boosts reflectance detail; larger c sharpens the transition; larger d0 pushes more frequencies into the high-gain region

# Log -> FFT -> filter -> inverse FFT -> exp
img_log = np.log1p(img.astype(np.float32))
F_log = np.fft.fftshift(np.fft.fft2(img_log))

H = (gamma_h - gamma_l) * (1 - np.exp(-c * (D**2) / (d0**2))) + gamma_l

img_filtered = np.real(np.fft.ifft2(np.fft.ifftshift(F_log * H)))
homomorphic = np.expm1(img_filtered)
homomorphic = cv.normalize(homomorphic, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)

show({"Original": img, "H": H, "Homomorphic Filter": homomorphic})